<!-- NB_DOC_INTRO_V1 -->
# ML — Bench multi-modeles + MLflow tracking

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Pattern **bench multi-modeles avec MLflow** : entrainer N modeles, log chaque run (params, metrics, modele), comparer dans le UI. Ce notebook execute un bench complet avec MLflow local (file:// store).

Pour MLflow + DVC + W&B comparatif : [MLOps_Tracking_DVC_Wandb.ipynb](./MLOps_Tracking_DVC_Wandb.ipynb).

## Plan

1. Setup MLflow (file store local)
2. Bench 4 modeles avec MLflow tracking
3. Comparer runs (search_runs)
4. Charger un modele depuis un run
5. Promotion via aliases (staging/production)
6. Pieges + References


In [ ]:
import mlflow
import numpy as np
import pandas as pd
import os, tempfile
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")
SEED = 42

# MLflow local (file://)
demo_dir = tempfile.mkdtemp(prefix="mlflow_bench_")
mlflow.set_tracking_uri(f"file:///{demo_dir.replace(os.sep, '/')}")
mlflow.set_experiment("ml_bench_demo")
print(f"MLflow URI : {mlflow.get_tracking_uri()}")

## 1. Bench de 4 modeles avec MLflow

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

models = {
    "LogReg":     Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))]),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "GBM":        GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    "SVC":        Pipeline([("sc", StandardScaler()), ("clf", SVC(probability=True, random_state=SEED))]),
}

run_ids = {}
for name, model in models.items():
    with mlflow.start_run(run_name=name) as run:
        # Log model name + params
        mlflow.log_param("model", name)
        if hasattr(model, "get_params"):
            for k, v in model.get_params(deep=False).items():
                try:
                    mlflow.log_param(k, str(v)[:100])
                except Exception:
                    pass

        # Train
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        # Metrics
        mlflow.log_metric("accuracy", accuracy_score(y_te, pred))
        mlflow.log_metric("f1", f1_score(y_te, pred))
        mlflow.log_metric("auc", roc_auc_score(y_te, proba))

        # Model
        mlflow.sklearn.log_model(model, "model")
        run_ids[name] = run.info.run_id
        print(f"  {name}: AUC={roc_auc_score(y_te, proba):.4f}, run_id={run.info.run_id[:8]}")

## 2. Comparer les runs

In [ ]:
exp = mlflow.get_experiment_by_name("ml_bench_demo")
runs_df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
cols = ["run_id", "tags.mlflow.runName", "metrics.accuracy", "metrics.f1", "metrics.auc"]
print(runs_df[cols].sort_values("metrics.auc", ascending=False).to_string(index=False))

## 3. Charger un modele depuis MLflow

In [ ]:
# Best run = highest AUC
best_run = runs_df.sort_values("metrics.auc", ascending=False).iloc[0]
best_name = best_run["tags.mlflow.runName"]
best_id = best_run["run_id"]
print(f"Best : {best_name} (id={best_id[:8]}, AUC={best_run['metrics.auc']:.4f})")

# Charger le modele
loaded = mlflow.sklearn.load_model(f"runs:/{best_id}/model")
pred = loaded.predict(X_te)
print(f"Reload acc : {accuracy_score(y_te, pred):.4f}")

## 4. Model Registry avec aliases (pattern moderne)

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

model_name = "demo_bc_classifier"
result = mlflow.register_model(model_uri=f"runs:/{best_id}/model", name=model_name)
print(f"Modele enregistre : {result.name} v{result.version}")

# Promouvoir via alias (pattern moderne, depuis MLflow 2.9+)
client.set_registered_model_alias(name=model_name, alias="production", version=result.version)
print(f"  → alias 'production' assigne a v{result.version}")

# Charger via alias
loaded = mlflow.sklearn.load_model(f"models:/{model_name}@production")
print(f"Loaded via alias 'production' : {type(loaded).__name__}")

## 5. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Pas de nom de run | `mlflow.start_run(run_name="...")` |
| Logger TOUT en autolog (lourd) | Selectif : `autolog(log_models=False)` |
| Pas de tags | `mlflow.set_tag("env", "prod")` |
| Promouvoir directement Production | Pipeline Staging → tests → Production |
| URL en dur | `MLFLOW_TRACKING_URI` env var partagee equipe |


## References

### Documentation
- MLflow : https://mlflow.org/docs/latest/index.html

### Voir aussi
- [MLOps_Tracking_DVC_Wandb.ipynb](MLOps_Tracking_DVC_Wandb.ipynb)
- [MLOps_Drift_Monitoring.ipynb](MLOps_Drift_Monitoring.ipynb)
- [MLOps_Pipelines_Airflow_Prefect.ipynb](MLOps_Pipelines_Airflow_Prefect.ipynb)
